In [3]:
import sys
import os

# 1. Xóa sạch thư mục cũ đi
!rm -rf /kaggle/working/drowsiness_detection_project

# 2. Tải bản mới từ GitHub
# LƯU Ý: Nhớ thay bằng link repo thực tế của bạn
!git clone https://github.com/KerryFT/drowsiness_detection_project.git

# 3. Đưa thư mục vào đường dẫn hệ thống
PROJECT_PATH = '/kaggle/working/drowsiness_detection_project'
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

# 4. XÓA SỔ CACHE BỘ NHỚ NGẦM (Cách triệt để 100%)
# Duyệt qua các module trong src và xóa chúng khỏi bộ nhớ tạm
modules_to_clear = ['src', 'src.dataset', 'src.models.mobilenet', 'src.engine', 'src.augmentations', 'src.config']
for mod in modules_to_clear:
    if mod in sys.modules:
        del sys.modules[mod]

# 5. Import lại một cách sạch sẽ
from src.dataset import get_dataloaders
from src.models.mobilenet import build_baseline_model
from src.engine import train_one_epoch, evaluate_one_epoch

print("✅ Đã dọn dẹp cache và tải code mới nhất thành công!")

Cloning into 'drowsiness_detection_project'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 46 (delta 13), reused 43 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 11.88 KiB | 1.48 MiB/s, done.
Resolving deltas: 100% (13/13), done.
✅ Đã dọn dẹp cache và tải code mới nhất thành công!


**Baseline: MobileNetV3-Small**

In [4]:
import torch
import torch.nn as nn

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EPOCHS = 10

print("Device:", DEVICE)

Device: cuda


In [5]:
import os
import glob

# Thay đường dẫn này bằng đường dẫn bạn đang dùng trong DATA_DIR
DATA_DIR = '/kaggle/input/datasets/minhngt02/uta-rldd'
train_dir = os.path.join(DATA_DIR, 'train')

print("1. Thư mục train có tồn tại không?", os.path.isdir(train_dir))

if os.path.isdir(train_dir):
    print("2. Các thư mục con bên trong train:")
    folders = os.listdir(train_dir)
    print(folders)
    
    # Thử vào thư mục đầu tiên xem ảnh có đuôi gì
    if len(folders) > 0:
        first_folder = os.path.join(train_dir, folders[0])
        files = os.listdir(first_folder)
        print(f"3. Thư mục {folders[0]} có {len(files)} files.")
        if len(files) > 0:
            print(f"   File ví dụ: {files[0]}")

1. Thư mục train có tồn tại không? True
2. Các thư mục con bên trong train:
['active', 'fatigue']
3. Thư mục active có 5862 files.
   File ví dụ: img_h_5760.jpg


In [6]:
# Unpack 3 biến thay vì 2
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=16)

model = build_baseline_model(num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

[/kaggle/input/datasets/minhngt02/uta-rldd/train] Đã tạo 2261 chuỗi (8 frames/chuỗi).
[/kaggle/input/datasets/minhngt02/uta-rldd/val] Đã tạo 454 chuỗi (8 frames/chuỗi).
[/kaggle/input/datasets/minhngt02/uta-rldd/test] Đã tạo 225 chuỗi (8 frames/chuỗi).


In [7]:
best_f1 = 0.0
best_model_path = 'baseline_best.pt'

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    # 1. Học trên tập Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    # 2. Thi thử trên tập Val
    val_loss, val_f1 = evaluate_one_epoch(model, val_loader, criterion, DEVICE)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")
    
    # Cập nhật Scheduler dựa trên điểm Val
    scheduler.step(val_f1)
    
    # Lưu lại trạng thái tốt nhất
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"🔥 Đã lưu Model tốt nhất (Val F1: {best_f1:.4f})")


--- Epoch 1/10 ---
Train Loss: 0.5743 | Val Loss: 0.8919 | Val Macro-F1: 0.7320
🔥 Đã lưu Model tốt nhất (Val F1: 0.7320)

--- Epoch 2/10 ---
Train Loss: 0.5081 | Val Loss: 0.7897 | Val Macro-F1: 0.6709

--- Epoch 3/10 ---
Train Loss: 0.4938 | Val Loss: 0.5903 | Val Macro-F1: 0.7659
🔥 Đã lưu Model tốt nhất (Val F1: 0.7659)

--- Epoch 4/10 ---
Train Loss: 0.4818 | Val Loss: 0.5711 | Val Macro-F1: 0.8608
🔥 Đã lưu Model tốt nhất (Val F1: 0.8608)

--- Epoch 5/10 ---
Train Loss: 0.4681 | Val Loss: 0.3236 | Val Macro-F1: 0.9626
🔥 Đã lưu Model tốt nhất (Val F1: 0.9626)

--- Epoch 6/10 ---
Train Loss: 0.4587 | Val Loss: 0.3724 | Val Macro-F1: 0.8608

--- Epoch 7/10 ---
Train Loss: 0.4541 | Val Loss: 0.5160 | Val Macro-F1: 0.8349

--- Epoch 8/10 ---
Train Loss: 0.4608 | Val Loss: 0.4472 | Val Macro-F1: 0.7536

--- Epoch 9/10 ---
Train Loss: 0.4418 | Val Loss: 0.4002 | Val Macro-F1: 0.8862

--- Epoch 10/10 ---
Train Loss: 0.4344 | Val Loss: 0.3809 | Val Macro-F1: 0.9581


In [13]:
print("\n=== ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ===")

# 1. Tải lại bộ trọng số (weights) của Epoch đạt điểm Val cao nhất
model.load_state_dict(torch.load(best_model_path))

# 2. Chấm điểm trên tập Test (Tập dữ liệu mô hình chưa từng gặp)
test_loss, test_f1 = evaluate_one_epoch(model, test_loader, criterion, DEVICE)

print(f"🎯 Test Loss: {test_loss:.4f}")
print(f"🎯 Test Macro-F1: {test_f1:.4f}")
# Báo cáo con số này vào Slide thuyết trình!


=== ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ===
🎯 Test Loss: 0.3515
🎯 Test Macro-F1: 0.9511


**MobileNetV3-Small + CBAM + Temporal Shift Module (TSM)**

In [15]:
import sys
import os

!rm -rf /kaggle/working/drowsiness_detection_project
!git clone https://github.com/KerryFT/drowsiness_detection_project.git

PROJECT_PATH = '/kaggle/working/drowsiness_detection_project'
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

modules_to_clear = ['src', 'src.dataset', 'src.models.mobilenet', 'src.models.cbam', 'src.models.tsm', 'src.engine', 'src.augmentations', 'src.config']
for mod in modules_to_clear:
    if mod in sys.modules:
        del sys.modules[mod]

print("✅ Đã dọn dẹp cache và tải code mới nhất thành công!")

Cloning into 'drowsiness_detection_project'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 46 (delta 13), reused 43 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 11.88 KiB | 3.96 MiB/s, done.
Resolving deltas: 100% (13/13), done.
✅ Đã dọn dẹp cache và tải code mới nhất thành công!


In [16]:
import torch
import torch.nn as nn
from src.dataset import get_dataloaders
from src.models.mobilenet import build_proposed_model
from src.engine import train_one_epoch, evaluate_one_epoch

DATA_DIR = '/kaggle/input/datasets/minhngt02/uta-rldd'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 15 # Nên tăng số Epoch lên 15 vì mô hình giờ phức tạp hơn (cần học Attention)

train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=16)

# KHỞI TẠO MÔ HÌNH TSM + CBAM
model = build_proposed_model(num_classes=2, n_segment=8).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_f1 = 0.0
best_model_path = 'proposed_best.pt'

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_f1 = evaluate_one_epoch(model, val_loader, criterion, DEVICE)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")
    
    scheduler.step(val_f1)
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"🔥 Đã lưu Model tốt nhất (Val F1: {best_f1:.4f})")

# ĐÁNH GIÁ TRÊN TẬP TEST
print("\n=== ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ===")
model.load_state_dict(torch.load(best_model_path))
test_loss, test_f1 = evaluate_one_epoch(model, test_loader, criterion, DEVICE)
print(f"🎯 Test Loss: {test_loss:.4f}")
print(f"🎯 Test Macro-F1: {test_f1:.4f}")

[/kaggle/input/datasets/minhngt02/uta-rldd/train] Đã tạo 2261 chuỗi (8 frames/chuỗi).
[/kaggle/input/datasets/minhngt02/uta-rldd/val] Đã tạo 454 chuỗi (8 frames/chuỗi).
[/kaggle/input/datasets/minhngt02/uta-rldd/test] Đã tạo 225 chuỗi (8 frames/chuỗi).

--- Epoch 1/15 ---
Train Loss: 0.6277 | Val Loss: 0.7280 | Val Macro-F1: 0.3333
🔥 Đã lưu Model tốt nhất (Val F1: 0.3333)

--- Epoch 2/15 ---
Train Loss: 0.5835 | Val Loss: 0.6997 | Val Macro-F1: 0.3333

--- Epoch 3/15 ---
Train Loss: 0.5738 | Val Loss: 0.5814 | Val Macro-F1: 0.4964
🔥 Đã lưu Model tốt nhất (Val F1: 0.4964)

--- Epoch 4/15 ---
Train Loss: 0.5634 | Val Loss: 0.5051 | Val Macro-F1: 0.7516
🔥 Đã lưu Model tốt nhất (Val F1: 0.7516)

--- Epoch 5/15 ---
Train Loss: 0.5513 | Val Loss: 0.7491 | Val Macro-F1: 0.4115

--- Epoch 6/15 ---
Train Loss: 0.5445 | Val Loss: 0.5426 | Val Macro-F1: 0.7834
🔥 Đã lưu Model tốt nhất (Val F1: 0.7834)

--- Epoch 7/15 ---
Train Loss: 0.5506 | Val Loss: 0.5456 | Val Macro-F1: 0.5981

--- Epoch 8/15 